# 23 — 2D Plate with an Edge Crack

**Curriculum slot:** Tier 2, slot 23.
**Prerequisite:** 01 — Hello Plate, 22 — Geometric selection.
**Strategy used for results:** vanilla `ops.recorder("mpco", ...)` + `Results.from_mpco` + viewer.

A rectangular plate with a horizontal edge crack on the left face, loaded in vertical tension on the top edge (mode-I). The mesh is refined around the crack tip via a distance field, gmsh's `Crack` plugin duplicates the crack-lip nodes (and the mouth, since it sits on a free edge), and the .mpco results are read back and displayed in the post-solve viewer.

This notebook intentionally keeps the apeGmsh side **verbose-by-name** (labels at creation, queries for boundary subsets, no raw entity tags) and the OpenSees side **vanilla openseespy** (explicit `ops.*` analysis loop and recorder declaration).

> **Note on the crack model.** The crack is modelled as a 1-D line embedded in the plate surface. `g.mesh.editing.crack` walks the elements on one side of the line and reconnects them to a duplicated set of nodes — the **lips and mouth open**, while the **tip stays shared** so the K-field is captured. The plate is also fragmented around the line so the mesh is conforming.

## 1. Imports and parameters

In [1]:
from pathlib import Path

import numpy as np
import openseespy.opensees as ops

from apeGmsh import apeGmsh, Results

# Geometry (consistent units: mm, MPa, N)
L  = 200.0     # plate length
H  = 100.0     # plate height
a  = 50.0      # edge crack length (from x=0 to x=a, on y = H/2)
t  = 1.0       # plate thickness (plane stress)

# Material — linear elastic, plane stress
E  = 210_000.0
NU = 0.3

# Loading — vertical tension on the top edge (perpendicular to the crack -> mode-I opens it)
SIGMA = 50.0       # tensile stress on the top face

# Mesh sizing — fine near the tip, coarser away
LC_FAR  = 6.0
LC_TIP  = 1.0
RAMP_R0 = 2.0      # inner radius of the refinement band
RAMP_R1 = 30.0     # outer radius — beyond this, size = LC_FAR

# Output
OUT_DIR  = Path("23_plate_with_crack_out")
OUT_DIR.mkdir(exist_ok=True)
MPCO_FILE = OUT_DIR / "plate_with_crack.mpco"

## 2. Geometry — plate + embedded crack

Every entity carries a `label=` at creation. The crack-tip point gets its own label so the refinement field can find it by name.

In [2]:
g = apeGmsh(model_name="23_plate_with_crack", verbose=False)
g.begin()

geo = g.model.geometry

# Plate
plate = geo.add_rectangle(0.0, 0.0, 0.0, L, H, label="Plate")

# Crack — line on y = H/2 from the left face to x = a, with named tip
p_mouth = geo.add_point(0.0, H / 2.0, 0.0, label="CrackMouth")
p_tip   = geo.add_point(a,   H / 2.0, 0.0, label="CrackTip")
crack   = geo.add_line(p_mouth, p_tip, label="Crack")

# Fragment splits the rectangle's left edge at the crack mouth and
# embeds the crack line into the plate as a conforming internal edge.
g.model.boolean.fragment([(2, plate)], [(1, crack)])

g.model.sync()

Model(name='23_plate_with_crack', entities=9)

## 3. Physical groups — by label and by query

Promote the labels to physical groups for `Plate`, `Crack`, and `CrackTip`. The four plate edges are picked by geometric predicate — no raw tags.

In [3]:
g.labels.promote_to_physical("Plate")
g.labels.promote_to_physical("Crack")
g.labels.promote_to_physical("CrackTip")
g.labels.promote_to_physical("CrackMouth")

edges = g.model.queries.boundary("Plate", dim=2, oriented=False)
g.model.queries.select(edges, on={"x": 0.0}).to_physical("Left")
g.model.queries.select(edges, on={"x": L  }).to_physical("Right")
g.model.queries.select(edges, on={"y": 0.0}).to_physical("Bottom")
g.model.queries.select(edges, on={"y": H  }).to_physical("Top")

Selection(1 curves) — .select(on=..., crossing=...) to filter further

## 4. Refinement field, mesh, and node duplication along the crack

Distance field anchored on the labelled CrackTip — element size ramps from LC_TIP near the tip to LC_FAR beyond RAMP_R1.

After meshing, gmsh's built-in Crack plugin (wrapped here as g.mesh.editing.crack) walks the elements on one side of the Crack physical group and reconnects them to a freshly duplicated set of nodes. The crack mouth is named in open_boundary so its node is duplicated too — the crack opens onto the free left edge — while the tip stays shared so the K-field is captured.

In [4]:
df = g.mesh.field.distance(
    points=g.labels.entities("CrackTip"),
)
tf = g.mesh.field.threshold(
    df,
    size_min=LC_TIP, size_max=LC_FAR,
    dist_min=RAMP_R0, dist_max=RAMP_R1,
)
g.mesh.field.set_background(tf)
g.mesh.sizing.set_global_size(LC_FAR)

g.mesh.generation.generate(dim=2)

# Duplicate nodes along the crack so the lips can separate.
# open_boundary names the PG of vertices that should ALSO be
# duplicated (the crack mouth on the free left edge); the tip
# is kept shared by default — that's where stress concentrates.
g.mesh.editing.crack(
    "Crack", dim=1, open_boundary="CrackMouth",
)

g.mesh.partitioning.renumber(dim=2, base=1)
fem = g.mesh.queries.get_fem_data(dim=2)
print(f"mesh built: {fem.info.n_nodes} nodes, {fem.info.n_elems} elements")

mesh built: 1184 nodes, 2231 elements


## 5. Build the OpenSees model — vanilla openseespy

Plane-stress `tri31` elements driven directly via `ops.*` calls. We pull node coordinates and element connectivity straight from the FEM broker, then apply the boundary conditions and the right-edge tension as nodal forces.

In [5]:
ops.wipe()
ops.model("basic", "-ndm", 2, "-ndf", 2)

# Nodes
for nid, xyz in fem.nodes.get():
    ops.node(int(nid), float(xyz[0]), float(xyz[1]))

# Material — plane stress elastic isotropic
MAT_TAG = 1
ops.nDMaterial("ElasticIsotropic", MAT_TAG, E, NU)

# Elements — one tri31 per mesh element on the Plate PG
for group in fem.elements.get(pg="Plate"):
    for eid, conn in zip(group.ids, group.connectivity):
        ops.element(
            "tri31", int(eid),
            int(conn[0]), int(conn[1]), int(conn[2]),
            t, "PlaneStress", MAT_TAG,
        )

# Boundary conditions — bottom edge is a roller in y, with one corner pinned in x.
# The cracked LEFT edge stays free so the lips/mouth can separate.
bottom_ids = list(fem.nodes.get(pg="Bottom").ids)
for n in bottom_ids:
    ops.fix(int(n), 0, 1)

# Pin x at one bottom corner to remove the rigid-body translation in x
# (y is already pinned by the roller above — only the x DOF gets added here).
bottom_coords = fem.nodes.get(pg="Bottom").coords
corner_idx = int(np.argmin(bottom_coords[:, 0]))
ops.fix(int(bottom_ids[corner_idx]), 1, 0)

# Top edge — vertical tension distributed equally among nodes
top_ids = list(fem.nodes.get(pg="Top").ids)
F_per_n = (SIGMA * L * t) / len(top_ids)

ops.timeSeries("Linear", 1)
ops.pattern("Plain", 1, 1)
for n in top_ids:
    ops.load(int(n), 0.0, F_per_n)

Tri31 - Written by Roozbeh G. Mikola and N.Sitar, UC Berkeley


## 6. Declare an MPCO recorder — vanilla `ops.recorder`

Single recorder, written next to the notebook. Nodes capture displacement and reactions; elements capture Cauchy stress and small-strain at the (single) Gauss point of each `tri31`.

In [6]:
if MPCO_FILE.exists():
    MPCO_FILE.unlink()

ops.recorder(
    "mpco", str(MPCO_FILE),
    "-N", "displacement", "reactionForce",
    "-E", "stresses", "strains",
    "-T", "dt", 1.0,
)

MPCO recorder - Written by ASDEA Software Technology: M.Petracca, G.Camata
ASDEA Software Technology: https://asdeasoft.net 
STKO (Scientific ToolKit for OpenSees): https://asdeasoft.net/stko/ 
If you use this tool, please cite us:
Petracca, M., Candeloro, F., & Camata, G. (2017). "STKO user manual". ASDEA Software Technology, Pescara Italy.


0

## 7. Run the analysis — vanilla openseespy static loop

Single load step ramped from 0 → 1 over 10 increments. Linear elastic, so this is overkill — but the increments give the viewer a time slider to scrub through.

In [7]:
N_STEPS = 10

ops.system("BandGeneral")
ops.numberer("RCM")
ops.constraints("Plain")
ops.integrator("LoadControl", 1.0 / N_STEPS)
ops.test("NormDispIncr", 1e-8, 25)
ops.algorithm("Newton")
ops.analysis("Static")

ok = ops.analyze(N_STEPS)
print("analyze() returned:", ok, "(0 = converged)")

analyze() returned:

 0 (0 = converged)


## 8. Read results back from the .mpco file

In [8]:
# Flush the recorder so the .mpco file is fully written before we open it.
ops.wipeAnalysis()
ops.remove("recorders")

results = Results.from_mpco(MPCO_FILE, fem=fem)
print(results.inspect.summary())

Results: 23_plate_with_crack_out\plate_with_crack.mpco
  FEM: 1184 nodes, 2231 elements (snapshot_id=f5222dff9c4fad18502b6b3bdf963841)
  Stages (1):
    - stage_0 (MODEL_STAGE[1]): steps=1, kind=transient


## 9. Quick numerical sanity check

The mode-I opening is the **vertical separation** of the upper and lower lips at the crack mouth — i.e. the difference between the two duplicated mouth nodes' `Uy`. We also report the global ranges of `Ux`/`Uy` for context.

In [9]:
uy_all = results.nodes.get(component="displacement_y", time=-1)
ux_all = results.nodes.get(component="displacement_x", time=-1)
print(f"global max |Uy| = {np.max(np.abs(uy_all.values)):.4e} mm")
print(f"global max |Ux| = {np.max(np.abs(ux_all.values)):.4e} mm")

# Crack-mouth opening: after the Crack plugin, two duplicated mesh nodes sit at
# (0, H/2). Locate them by FEM coordinate, then look up Uy by node id.
all_ids    = np.asarray([int(i) for i in fem.nodes.get().ids])
all_coords = fem.nodes.get().coords
mouth_mask = (np.abs(all_coords[:, 0]) < 1e-6) & (np.abs(all_coords[:, 1] - H / 2.0) < 1e-6)
mouth_ids  = all_ids[mouth_mask]
print(f"mouth node count = {len(mouth_ids)} (expect 2 — upper + lower lip)")

if len(mouth_ids) == 2:
    slab_ids = np.asarray([int(i) for i in uy_all.node_ids])
    last_uy  = np.asarray(uy_all.values)[-1]    # (N,) at the final step
    id_to_uy = dict(zip(slab_ids, last_uy))
    uy0, uy1 = id_to_uy[int(mouth_ids[0])], id_to_uy[int(mouth_ids[1])]
    print(f"crack-mouth opening |Δ Uy| = {abs(uy0 - uy1):.4e} mm")

global max |Uy| = 1.6785e-02 mm
global max |Ux| = 6.7358e-03 mm
mouth node count = 2 (expect 2 — upper + lower lip)
crack-mouth opening |Δ Uy| = 1.6329e-02 mm


## 10. Open the post-solve viewer

The viewer shows the deformed plate with the K-field at the tip. Toggle through `displacement_x/y` and `stress.cauchy.sigma_xx/yy` in the Layer stack to see the crack-tip stress concentration.

In [10]:
import os
if os.environ.get("APEGMSH_NO_VIEWER") != "1":
    results.viewer()

## What this unlocks

* **Conforming embedded curves via `boolean.fragment`** — the rectangle is split at the crack line so the mesh respects it without manual surface partitioning.
* **Discontinuity via `g.mesh.editing.crack`** — gmsh's `Crack` plugin duplicates lip nodes (and the mouth via `open_boundary=`) while keeping the tip shared. Geometry stays single, mesh becomes two-sided.
* **Distance-anchored refinement** by labelled point — the same idiom works for stress concentrations near holes, fillets, or any geometric singularity.
* **Verbose-by-name throughout the apeGmsh side** — every entity is named once at creation, then referenced by label/PG; queries (`select(edges, on={...})`) replace raw tags for boundary subsets.
* **Vanilla openseespy interop** — the FEM broker (`fem.nodes.*`, `fem.elements.*`) is just data; feed it straight to `ops.*` for full control of the analysis loop and recorder syntax.

In [11]:
g.end()